# Continuar el entrenamiento en Colab desde VS Code

Selecciona **Select Kernel → Colab → New Colab Server → GPU (T4 si está disponible)**.
Inicia sesión en Google cuando se solicite. Ejecutaremos las celdas por orden.
El cálculo se realiza en Google; este notebook se abre y controla desde VS Code.

1. Comprobar GPU.
2. Subir `data/colab/upload_parts` desde el Explorador de VS Code con **Upload to Colab** y preparar el entorno.
3. Entrenar 20 épocas nuevas en GPU desde los mismos pesos preentrenados, conservando 640 píxeles y los splits originales. El checkpoint del Mac se conserva aparte tras una incompatibilidad del optimizador al reanudar.
4. Recuperar resultados al terminar. El test sigue reservado.


In [1]:
import sys
import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Selecciona un servidor Colab con GPU antes de continuar."
print("GPU:", torch.cuda.get_device_name(0))
print("Memoria GPU (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


Python: 3.13.16
PyTorch: 2.11.0+cu130
GPU: Tesla T4
Memoria GPU (GB): 14.6


## Preparar los archivos y las dependencias
Antes de ejecutar esta celda, sube **data/colab/upload_parts** al servidor con **Upload to Colab** desde el Explorador de VS Code.
El archivo contiene el dataset público, el código y los checkpoints. La clave de Roboflow se conserva en el Mac.


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, subprocess, sys, zipfile

assert torch.cuda.is_available(), "Conecta una GPU antes de preparar el entrenamiento."
archive = Path("/content/colab_upload.zip")
parts = sorted(Path("/content/upload_parts").glob("colab_upload.part*"))
if parts:
    assert len(parts) == 27, f"La subida sigue en curso: {len(parts)}/27 partes."
    with archive.open("wb") as target:
        for part in parts:
            target.write(part.read_bytes())
assert archive.is_file(), "Sube data/colab/upload_parts al servidor desde el Explorador de VS Code."
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '230fbd712c919cf40a1b83ffc5d97bbaa3de7058c39f0c311b6574697662f061', "Subida incompleta o distinta del archivo verificado."
cloud_root = Path("/content/car_damage_assignment")
assert not (cloud_root / "cloud_prepared.json").exists(), "El entorno ya está preparado; continúa con la siguiente celda."
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert Path("/content") in (Path("/content") / name).resolve().parents
    z.extractall("/content")
manifest = json.loads((cloud_root / "transfer_manifest.json").read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((cloud_root / name).read_bytes()).hexdigest() == expected, name
print("Archivos verificados. Instalando la misma versión de Ultralytics...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ultralytics==8.4.173"], check=True)
import importlib.metadata
print("Ultralytics:", importlib.metadata.version("ultralytics"))
segment = {
    "prepared_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version, "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda, "ultralytics": importlib.metadata.version("ultralytics"),
    "saved_epochs": 3, "target_total_epochs": 20, "batch": 8, "workers": 2,
    "image_size": 640, "test_evaluated": False,
    "checkpoint_sha256": manifest["runs/detect/yolo11n_20epochs_20261025T102112Z/weights/last.pt"],
}
(cloud_root / "results/yolo11n_20epochs_20261025T102112Z/colab_segment.json").write_text(json.dumps(segment, indent=2))
(cloud_root / "cloud_prepared.json").write_text(json.dumps(segment, indent=2))
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(cloud_root / "results/yolo11n_20epochs_20261025T102112Z/colab_packages.txt").write_text(freeze)
print("Preparado: continuaremos desde 3 épocas guardadas hasta 20.")


Archivos verificados. Instalando la misma versión de Ultralytics...
Ultralytics: 8.4.173
Preparado: continuaremos desde 3 épocas guardadas hasta 20.


## Entrenar las 20 épocas en GPU
Esta celda inicia un experimento nuevo de 20 épocas mediante **main.py train** en la GPU de Colab. Se verifican por SHA256 los mismos pesos preentrenados de la primera prueba. Déjala abierta hasta que termine.
La selección del modelo usa validación. Se mantienen el dataset y sus etiquetas originales.


In [4]:
import os, subprocess, sys
from pathlib import Path
cloud_root = Path("/content/car_damage_assignment")
assert (cloud_root / "cloud_prepared.json").is_file(), "Ejecuta primero la preparación."
# A new CUDA experiment avoids the Mac checkpoint's incompatible optimizer state.
import hashlib, json
from ultralytics.utils.downloads import attempt_download_asset
pretrained = cloud_root / "data/pretrained/yolo11n.pt"
pretrained.parent.mkdir(parents=True, exist_ok=True)
pretrained = Path(attempt_download_asset(str(pretrained)))
assert hashlib.sha256(pretrained.read_bytes()).hexdigest() == '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1', "Los pesos preentrenados no coinciden con los del primer experimento."
config_path = cloud_root / "training_config.json"
config = json.loads(config_path.read_text())
config.update({"resume_from": None, "experiment_name": "yolo11n_20epochs_colab", "status": "ready_for_colab"})
config["notes"].append("Start a fresh 20-epoch CUDA experiment after the MPS checkpoint optimizer failed with a fused Adam dtype/device/layout mismatch. Preserve the Mac checkpoint separately. The fresh run uses exactly the original pretrained weights, checked by SHA256.")
config_path.write_text(json.dumps(config, indent=2))
print("Nuevo experimento: 20 épocas en GPU desde los mismos pesos preentrenados.", flush=True)
log_path = cloud_root / "colab_training.log"
with log_path.open("a") as log:
    process = subprocess.Popen([sys.executable, "-u", "main.py", "train"], cwd=cloud_root,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return_code = process.wait()
    except KeyboardInterrupt:
        import signal
        os.killpg(process.pid, signal.SIGINT)
        process.wait()
        raise
if return_code:
    raise RuntimeError(f"El entrenamiento terminó con código {return_code}. Consulta la salida anterior.")
import shutil
state = json.loads((cloud_root / "results/latest_training.json").read_text())
result_dir = cloud_root / state["directory"]
segment = json.loads((cloud_root / "cloud_prepared.json").read_text())
segment["source_mac_checkpoint_saved_epochs"] = segment.pop("saved_epochs")
segment.update({"saved_epochs_at_start": 0, "resume_applied": False, "pretrained_sha256": '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1',
                "migration_note": "Fresh CUDA experiment after MPS optimizer incompatibility; Mac checkpoint preserved separately."})
(result_dir / "colab_segment.json").write_text(json.dumps(segment, indent=2))
shutil.copy2(log_path, result_dir / "colab_training.log")
print("Entrenamiento completado. Ejecuta la celda siguiente para preparar la descarga.")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Nuevo experimento: 20 épocas en GPU desde los mismos pesos preentrenados.
Using the existing local dataset export.
PILOT device=0; images={'train': 1501, 'valid': 366}; test_evaluated=False
New https://pypi.org/project/ultralytics/8.4.188 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/

## Recuperar el modelo y los resultados
Ejecuta esta celda cuando finalice el entrenamiento. Después descarga **/content/colab_results.zip** desde la sección **Colab → Contents** de VS Code.
Se incluyen también los checkpoints para poder conservar el progreso si el entrenamiento se interrumpe.


In [5]:
import zipfile
from pathlib import Path
cloud_root = Path("/content/car_damage_assignment")
result_zip = Path("/content/colab_results.zip")
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for directory in (cloud_root / "results", cloud_root / "weights", cloud_root / "runs"):
        if directory.exists():
            for path in sorted(directory.rglob("*")):
                if path.is_file():
                    z.write(path, str(path.relative_to(cloud_root)))
    z.write(cloud_root / "training_config.json", "training_config.json")
print("Descarga preparada:", result_zip, "—", round(result_zip.stat().st_size / 1024**2, 1), "MB")


Descarga preparada: /content/colab_results.zip — 65.3 MB
